# 02 - Preprocessing Experiments: Does Domain-Aware NLP Actually Help?

`src/preprocessor.py` protects ~40 "financial signal words" (bullish,
bearish, downgrade, rally, ...) from stop-word removal, on the assumption
that a generic NLTK pipeline would otherwise strip them and lose sentiment
signal. Notebook 01 confirmed these words *do* correlate with sentiment -
but does generic preprocessing actually remove them?

This notebook tests that assumption directly by building a "naive" pipeline
(plain NLTK stop words, no protection) and comparing it side-by-side with
`FinancialTextPreprocessor`. The results were not quite what the original
design assumed - and surfaced two real gaps worth documenting.

In [1]:
import sys, re
from pathlib import Path

from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer

sys.path.insert(0, str(Path.cwd().parent / "src"))
from preprocessor import FinancialTextPreprocessor, FINANCIAL_SIGNAL_WORDS

lemmatizer  = WordNetLemmatizer()
naive_stops = set(stopwords.words("english"))
ours        = FinancialTextPreprocessor()

def naive_clean(text: str) -> str:
    """A 'default' NLTK pipeline with no domain-specific overrides."""
    text   = text.lower()
    text   = re.sub(r"[^\w\s$%]", " ", text)
    tokens = word_tokenize(text)
    tokens = [t for t in tokens if t not in naive_stops and len(t) > 1]
    tokens = [lemmatizer.lemmatize(t) for t in tokens]
    return " ".join(tokens)

## Experiment 1 - were the "protected" words ever actually at risk?

`FinancialTextPreprocessor` computes
`EFFECTIVE_STOP_WORDS = NLTK_STOPWORDS - FINANCIAL_SIGNAL_WORDS - {no, not, nor}`.
This only changes behaviour for words that are in **both** NLTK's stopword
list **and** our protected set. Let's check the actual overlap.

In [2]:
overlap = FINANCIAL_SIGNAL_WORDS & naive_stops
print(f"Financial signal words ({len(FINANCIAL_SIGNAL_WORDS)} total) "
      f"that are ALSO NLTK stopwords: {overlap}")
print(f"\n'no'/'not'/'nor' in NLTK stopwords: "
      f"{({'no','not','nor'} & naive_stops)}")

Financial signal words (53 total) that are ALSO NLTK stopwords: set()

'no'/'not'/'nor' in NLTK stopwords: {'no', 'not', 'nor'}


**Finding 1 - the protection is a no-op for the words it names.** None of
"bullish", "bearish", "downgrade", "rally", "profit", etc. were ever in
NLTK's stopword list - they're ordinary content words, not stop words. Listing
them in `FINANCIAL_SIGNAL_WORDS` doesn't *hurt* anything, but it also isn't
preventing the failure mode it was designed to prevent, because that failure
mode doesn't occur for these words.

The thing that *is* in NLTK's stopword list - and that the same line of code
*does* protect - is `{'no', 'not', 'nor'}`. That turns out to be the part
that matters. Let's look at why.

## Experiment 2 - negation: the protection that actually does something

Bag-of-words models (TF-IDF + LogReg/SVM) represent "the company did **not**
profit" and "the company **recorded a** profit" by which words are present,
not their order. If "not" is stripped as a stopword, these two sentences -
one negative, one positive - become nearly indistinguishable.

In [3]:
pairs = [
    "The company did not profit this quarter",
    "The company recorded a profit this quarter",
]

for text in pairs:
    print("IN   :", text)
    print("NAIVE:", naive_clean(text))
    print("OURS :", ours.clean(text))
    print()

IN   : The company did not profit this quarter
NAIVE: company profit quarter
OURS : company not profit quarter

IN   : The company recorded a profit this quarter
NAIVE: company recorded profit quarter
OURS : company recorded profit quarter



**Finding 2 - naive preprocessing collapses these two sentences to nearly
identical bags of words** (`company profit quarter` vs.
`company recorded profit quarter`) - a TF-IDF model sees them as
overwhelmingly similar despite opposite meanings. My pipeline keeps "not",
producing `company not profit quarter`. With the `ngram_range=(1,2)` setting
in `classical_models.py`, this also makes the bigram **"not profit"**
available as a feature - a token combination a naive pipeline can never
produce, because "not" doesn't survive to form a bigram.

This is the real, demonstrable benefit of the `EFFECTIVE_STOP_WORDS`
construction - it just isn't the benefit the variable name
(`FINANCIAL_SIGNAL_WORDS`) suggests.

## Experiment 3 - what gets silently lost that *should* be protected?

If negation words are the real win, are there other directionally-important
words that NLTK treats as stopwords and that I *haven't* protected? "down"
is a natural candidate - "shares were down 5%" is about as financially
meaningful as it gets.

In [4]:
print("Is 'down' an NLTK stopword?", "down" in naive_stops)
print("Is 'down' in FINANCIAL_SIGNAL_WORDS?", "down" in FINANCIAL_SIGNAL_WORDS)
print()

text = "Shares were down sharply after the earnings miss"
print("IN   :", text)
print("NAIVE:", naive_clean(text))
print("OURS :", ours.clean(text))

Is 'down' an NLTK stopword? True
Is 'down' in FINANCIAL_SIGNAL_WORDS? False

IN   : Shares were down sharply after the earnings miss
NAIVE: share sharply earnings miss
OURS : share sharply earnings miss


**Finding 3 - "down" is lost, by both pipelines.** It's an NLTK stopword
and was never added to `FINANCIAL_SIGNAL_WORDS`, so it gets removed exactly
like "the" or "after". `"shares were down sharply"` and
`"shares were up sharply"` would preprocess to the *same* bag of words
(`share sharply...`), losing the one word that actually carries the
sentiment. **Recommended fix for `FINANCIAL_SIGNAL_WORDS`:** add directional
words - `down`, `up`, `above`, `below`, `under`, `over` - none of which are
in the current set.

## Experiment 4 — currency amounts

The preprocessing regex keeps `$` and `%` characters on the assumption that
magnitudes like "$2.5B" or "fell 8%" carry signal. Does that survive
tokenization intact?

In [5]:
text = "The stock rose $3 to $45 after the upgrade"
print("IN   :", text)
print("OURS :", ours.clean(text))

# Inspect what word_tokenize does to '$3' specifically
import re
stripped = re.sub(r"[^\w\s$%]", " ", text.lower())
print("\nAfter punctuation strip:", repr(stripped))
print("After word_tokenize       :", word_tokenize(stripped))

IN   : The stock rose $3 to $45 after the upgrade
OURS : stock rose 45 upgrade

After punctuation strip: 'the stock rose $3 to $45 after the upgrade'
After word_tokenize       : ['the', 'stock', 'rose', '$', '3', 'to', '$', '45', 'after', 'the', 'upgrade']


**Finding 4 - single-digit dollar amounts disappear entirely.**
`word_tokenize` splits `$3` into the two tokens `$` and `3`. Both have
`len(t) == 1`, so the `len(t) > 1` filter (intended to drop noise like stray
punctuation) removes both - "$3" vanishes completely, while "$45" survives
as "45" (losing the `$` but keeping the magnitude). The sentence "rose $3 to
$45" preprocesses to `stock rose 45 upgrade` - the starting price is gone,
only the ending price remains. **Idea :** tokenize currency amounts
as a single unit (e.g. a regex pass that converts `$3` -> `usd3` *before*
tokenization) rather than relying on the generic tokenizer.

## Summary

Three concrete findings, none of which were obvious from reading
`preprocessor.py` in isolation:

1. The `FINANCIAL_SIGNAL_WORDS` protection list, as named, is mostly a no-op
   — those words were never NLTK stopwords.
2. Its real effect - preserving `no`/`not`/`nor` - *does* meaningfully change
   how negated sentences are represented, which matters for a bag-of-words
   model.
3. Two gaps exist that the current pipeline doesn't handle: directional words
   like "down"/"up" are stripped as ordinary stopwords, and single-digit
   dollar amounts are destroyed by tokenization + the `len(t) > 1` filter.
